1. Khai báo thư viện

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

2. Đọc dữ liệu

In [2]:
DATA_DIR = "../data/raw/"

customers = pd.read_csv(f"{DATA_DIR}customers.csv")
geography = pd.read_csv(f"{DATA_DIR}geography.csv")
inventory = pd.read_csv(f"{DATA_DIR}inventory.csv")
order_items = pd.read_csv(f"{DATA_DIR}order_items.csv")
orders = pd.read_csv(f"{DATA_DIR}orders.csv")
payments = pd.read_csv(f"{DATA_DIR}payments.csv")
products = pd.read_csv(f"{DATA_DIR}products.csv")
promotions = pd.read_csv(f"{DATA_DIR}promotions.csv")
returns = pd.read_csv(f"{DATA_DIR}returns.csv")
reviews = pd.read_csv(f"{DATA_DIR}reviews.csv")
shipments = pd.read_csv(f"{DATA_DIR}shipments.csv")
sales = pd.read_csv(f"{DATA_DIR}sales.csv")
web_traffic = pd.read_csv(f"{DATA_DIR}web_traffic.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}sample_submission.csv")


print("Đã đọc xong")

C:\Users\OS\AppData\Local\Temp\ipykernel_35496\147972244.py:6: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(f"{DATA_DIR}order_items.csv")


Đã đọc xong


3. Chuyển đổi định dạng d

In [3]:
orders['order_date'] = pd.to_datetime(orders['order_date'])
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])
promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date'] = pd.to_datetime(promotions['end_date'])
returns['return_date'] = pd.to_datetime(returns['return_date'])
reviews['review_date'] = pd.to_datetime(reviews['review_date'])
shipments['ship_date'] = pd.to_datetime(shipments['ship_date'])
shipments['delivery_date'] = pd.to_datetime(shipments['delivery_date'])
sales['Date'] = pd.to_datetime(sales['Date'])
web_traffic['date'] = pd.to_datetime(web_traffic['date'])

print("Chuyển đổi định dạng datetime")

Chuyển đổi định dạng datetime


4. Làm sạch dữ liệu và xử lí null

In [4]:
order_items['promo_id'] = order_items['promo_id'].fillna('NO_PROMO')
order_items['promo_id_2'] = order_items['promo_id_2'].fillna('NO_PROMO')
order_items['discount_amount'] = order_items['discount_amount'].fillna(0.0)

promotions['applicable_category'] = promotions['applicable_category'].fillna('All')

orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()

print("Dữ liệu đã được làm sạch và xử lý null")

Dữ liệu đã được làm sạch và xử lý null


5. Me

In [5]:
sales_master = pd.merge(order_items, orders, on='order_id', how='left')
sales_master = pd.merge(sales_master, products, on='product_id', how='left')

customers_no_zip = customers.drop(columns=['zip'])
sales_master = pd.merge(sales_master, customers_no_zip, on='customer_id', how='left')

sales_master = pd.merge(sales_master, geography, on='zip', how='left', suffixes=('', '_geo'))

sales_master = pd.merge(sales_master, shipments, on='order_id', how='left')

sales_master = pd.merge(sales_master, returns, on=['order_id', 'product_id'], how='left')

sales_master['return_quantity'] = sales_master['return_quantity'].fillna(0)
sales_master['refund_amount'] = sales_master['refund_amount'].fillna(0)

reviews_clean = reviews.drop(columns=['customer_id'])
sales_master = pd.merge(sales_master, reviews_clean, on=['order_id', 'product_id'], how='left')

sales_master['net_revenue'] = (sales_master['unit_price'] * sales_master['quantity']) - sales_master['discount_amount']
sales_master['final_revenue'] = sales_master['net_revenue'] - sales_master['refund_amount']

print(f"Kích thước bảng Master Sales: {sales_master.shape}")

Kích thước bảng Master Sales: (714673, 43)


6. Tính toán các chỉ số đặc trưng

In [6]:
sales_master['net_revenue'] = (sales_master['unit_price'] * sales_master['quantity']) - sales_master['discount_amount']

sales_master['gross_profit'] = sales_master['net_revenue'] - (sales_master['cogs'] * sales_master['quantity'])

sales_master[['order_date', 'product_name', 'net_revenue', 'gross_profit']].head()

,order_date,product_name,net_revenue,gross_profit
0,2012-07-04,VietMotion YY-09,7967.54,590.953941
1,2012-07-04,SaigonFlex UC-74,71163.75,8249.820384
2,2012-07-04,SaigonFlex UM-01,33660.99,3387.953233
3,2012-07-04,SaigonFlex UC-00,53196.25,7169.097610
4,2012-07-06,UrbanVN RP-10,1597.84,549.143643


In [7]:
web_traffic = web_traffic.rename(columns={'date': 'Date'})

daily_traffic = web_traffic.groupby('Date').agg({
    'sessions': 'sum',
    'unique_visitors': 'sum',
    'page_views': 'sum',
    'avg_session_duration_sec': 'mean'
}).reset_index()

daily_master = pd.merge(sales, daily_traffic, on='Date', how='left')

daily_master = daily_master.fillna(0)

print(f"Kích thước bảng Daily Master (Time-series): {daily_master.shape}")
daily_master.head()

Kích thước bảng Daily Master (Time-series): (3833, 7)


,Date,Revenue,COGS,sessions,unique_visitors,page_views,avg_session_duration_sec
0,2012-07-04,5123547.94,3982991.19,0.0,0.0,0.0,0.0
1,2012-07-05,2751773.45,2150580.23,0.0,0.0,0.0,0.0
2,2012-07-06,3054029.42,2517632.84,0.0,0.0,0.0,0.0
3,2012-07-07,2667930.94,2108246.62,0.0,0.0,0.0,0.0
4,2012-07-08,2360851.90,1808622.79,0.0,0.0,0.0,0.0


In [8]:
sales_master.info()

sales_master.describe()

<class 'pandas.DataFrame'>
RangeIndex: 714673 entries, 0 to 714672
Data columns (total 44 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             714673 non-null  int64         
 1   product_id           714673 non-null  int64         
 2   quantity             714673 non-null  int64         
 3   unit_price           714673 non-null  float64       
 4   discount_amount      714673 non-null  float64       
 5   promo_id             714673 non-null  str           
 6   promo_id_2           714673 non-null  str           
 7   order_date           714673 non-null  datetime64[us]
 8   customer_id          714673 non-null  int64         
 9   zip                  714673 non-null  int64         
 10  order_status         714673 non-null  str           
 11  payment_method       714673 non-null  str           
 12  device_type          714673 non-null  str           
 13  order_source         7146

,order_id,product_id,quantity,unit_price,discount_amount,order_date,customer_id,zip,price,cogs,...,delivery_date,shipping_fee,return_date,return_quantity,refund_amount,review_date,rating,net_revenue,final_revenue,gross_profit
count,714673.000000,714673.000000,714673.000000,714673.000000,714673.000000,714673,714673.000000,714673.000000,714673.000000,714673.000000,...,625386,625386.000000,39943,714673.000000,714673.000000,113553,113553.000000,714673.000000,714673.000000,714673.000000
mean,411615.839203,1234.933004,4.495989,5114.680265,1048.883057,2016-11-05 03:38:58.204466,85708.103028,55905.612497,5508.175172,4408.029863,...,2016-11-04 02:24:15.114122,4.839850,2016-11-11 23:32:53.362041,0.153358,714.537025,2016-11-13 07:48:27.600856,3.936021,21941.436386,21226.899360,2123.286293
min,1.000000,1.000000,1.000000,392.570000,0.000000,2012-07-04 00:00:00,1.000000,1001.000000,440.370000,249.280605,...,2012-07-06 00:00:00,0.000000,2012-07-11 00:00:00,0.000000,0.000000,2012-07-10 00:00:00,1.000000,389.740000,-17624.410000,-75957.121208
25%,203230.000000,689.000000,2.000000,1906.890000,0.000000,2014-07-22 00:00:00,42124.000000,31329.000000,2032.929931,1690.347319,...,2014-07-26 00:00:00,0.860000,2014-08-05 00:00:00,0.000000,0.000000,2014-08-07 00:00:00,3.000000,6041.630000,5648.930000,-173.929302
50%,409306.000000,990.000000,4.000000,4257.770000,0.000000,2016-06-29 00:00:00,89745.000000,54901.000000,4487.591719,3691.359645,...,2016-07-02 00:00:00,1.720000,2016-07-06 00:00:00,0.000000,0.000000,2016-07-14 00:00:00,4.000000,14518.070000,13699.580000,1116.906089
75%,618983.000000,2045.000000,6.000000,7273.750000,967.610000,2018-07-31 00:00:00,133840.000000,83860.000000,7960.852541,6445.037001,...,2018-07-29 00:00:00,2.590000,2018-08-08 00:00:00,0.000000,0.000000,2018-08-08 00:00:00,5.000000,30635.550000,29749.150000,4041.115185
max,834397.000000,2412.000000,8.000000,43056.000000,35235.470000,2022-12-31 00:00:00,157563.000000,99950.000000,40950.000000,38902.500000,...,2022-12-31 00:00:00,32.000000,2022-12-31 00:00:00,8.000000,160937.940000,2022-12-31 00:00:00,5.000000,331570.400000,331570.400000,80311.606561
std,240480.116484,691.332061,2.290145,3774.813457,2280.525121,NaN,48503.127919,28965.407199,3989.314155,3418.125708,...,NaN,8.754999,NaN,0.764267,4441.104519,NaN,1.149861,21712.261189,21549.611052,7575.220323


8. Xuất fi

In [10]:
import os

output_dir = "../data/processed/"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

data_to_export = {
    "customers_clean": customers,
    "geography_clean": geography,
    "inventory_clean": inventory,
    "order_items_clean": order_items,
    "orders_clean": orders,
    "payments_clean": payments,
    "products_clean": products,
    "promotions_clean": promotions,
    "returns_clean": returns,
    "reviews_clean": reviews,
    "sales_clean": sales,
    "shipments_clean": shipments,
    "web_traffic_clean": web_traffic,
    "sample_submission_clean": sample_sub,
    "sales_master_detail": sales_master
}


for file_name, df in data_to_export.items():
    path = f"{output_dir}{file_name}.csv"
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f" -> Đã lưu: {path}")

daily_master.to_csv(f"{output_dir}daily_master_timeseries.csv", index=False, encoding='utf-8-sig')

print("\n--- HOÀN THÀNH ---")
print(f"Tổng cộng đã xuất {len(data_to_export) + 1} tệp vào thư mục '{output_dir}'.")

 -> Đã lưu: ../data/processed/customers_clean.csv
 -> Đã lưu: ../data/processed/geography_clean.csv
 -> Đã lưu: ../data/processed/inventory_clean.csv
 -> Đã lưu: ../data/processed/order_items_clean.csv
 -> Đã lưu: ../data/processed/orders_clean.csv
 -> Đã lưu: ../data/processed/payments_clean.csv
 -> Đã lưu: ../data/processed/products_clean.csv
 -> Đã lưu: ../data/processed/promotions_clean.csv
 -> Đã lưu: ../data/processed/returns_clean.csv
 -> Đã lưu: ../data/processed/reviews_clean.csv
 -> Đã lưu: ../data/processed/sales_clean.csv
 -> Đã lưu: ../data/processed/shipments_clean.csv
 -> Đã lưu: ../data/processed/web_traffic_clean.csv
 -> Đã lưu: ../data/processed/sample_submission_clean.csv
 -> Đã lưu: ../data/processed/sales_master_detail.csv

--- HOÀN THÀNH ---
Tổng cộng đã xuất 16 tệp vào thư mục '../data/processed/'.
